# Step 0: NetworkX Baseline (Amazon0302)

This notebook computes **eigenvector centrality** using the precomputed **CSR graph files**.

CSR input folder: `dataset/amazon0302_csr/`

In [2]:
from pathlib import Path
from time import perf_counter
import json

import networkx as nx
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigs

CSR_DIR = Path('dataset/amazon0302_csr')
METADATA_PATH = CSR_DIR / 'metadata.json'
INDPTR_PATH = CSR_DIR / 'indptr.txt'
INDICES_PATH = CSR_DIR / 'indices.txt'
DATA_PATH = CSR_DIR / 'data.txt'

OUTPUT_DIR = Path('baseline/networkx')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_ITER = 1000
TOL = 1e-6
TOP_K = 20

ModuleNotFoundError: No module named 'networkx'

In [ ]:
assert CSR_DIR.exists(), f'Missing CSR directory: {CSR_DIR}'
assert METADATA_PATH.exists(), f'Missing metadata file: {METADATA_PATH}'
assert INDPTR_PATH.exists(), f'Missing indptr file: {INDPTR_PATH}'
assert INDICES_PATH.exists(), f'Missing indices file: {INDICES_PATH}'
assert DATA_PATH.exists(), f'Missing data file: {DATA_PATH}'

with METADATA_PATH.open('r', encoding='utf-8') as f:
    metadata = json.load(f)

num_nodes = int(metadata['num_nodes'])
num_edges = int(metadata['num_edges'])

indptr = np.loadtxt(INDPTR_PATH, dtype=np.int64)
indices = np.loadtxt(INDICES_PATH, dtype=np.int64)
data = np.loadtxt(DATA_PATH, dtype=np.float64)

A = csr_matrix((data, indices, indptr), shape=(num_nodes, num_nodes))
G = nx.from_scipy_sparse_array(A, create_using=nx.DiGraph)

print(f'Loaded CSR graph: nodes={num_nodes}, edges={num_edges}, nnz={A.nnz}')

In [ ]:
t0 = perf_counter()
converged = True

try:
    centrality = nx.eigenvector_centrality(
        G,
        max_iter=MAX_ITER,
        tol=TOL,
        weight=None
    )
except nx.PowerIterationFailedConvergence as e:
    converged = False
    centrality = getattr(e, 'eigenvector', None)
    if centrality is None:
        raise RuntimeError(
            f'Eigenvector centrality did not converge in {MAX_ITER} iterations and no partial vector was returned.'
        )

runtime_seconds = perf_counter() - t0
print(f'Centrality computed in {runtime_seconds:.3f} seconds | converged={converged}')

In [ ]:
scores_df = (
    pd.DataFrame(centrality.items(), columns=['node_id', 'score'])
    .sort_values('score', ascending=False)
    .reset_index(drop=True)
)

topk_df = scores_df.head(TOP_K).copy()
topk_df.insert(0, 'rank', range(1, len(topk_df) + 1))

scores_path = OUTPUT_DIR / 'amazon0302_eigenvector_scores.csv'
topk_path = OUTPUT_DIR / 'amazon0302_top20.csv'
metrics_path = OUTPUT_DIR / 'step0_metrics.json'

scores_df.to_csv(scores_path, index=False)
topk_df.to_csv(topk_path, index=False)

metrics = {
    'dataset': str(CSR_DIR),
    'num_nodes': int(num_nodes),
    'num_edges': int(num_edges),
    'method': 'networkx.eigenvector_centrality',
    'max_iter': int(MAX_ITER),
    'tol': float(TOL),
    'runtime_seconds': float(runtime_seconds),
    'converged': bool(converged)
}

with metrics_path.open('w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

print(f'Saved: {scores_path}')
print(f'Saved: {topk_path}')
print(f'Saved: {metrics_path}')

In [ ]:
display(topk_df)
metrics

## Optional: dominant eigenvalue from CSR adjacency

This computes the largest-magnitude eigenvalue directly from the CSR matrix `A`.

In [ ]:
eigvals, eigvecs = eigs(A, k=1, which='LR')  # Largest real part
dominant_eigenvalue = eigvals[0]
print('Dominant eigenvalue (complex form):', dominant_eigenvalue)
print('Dominant eigenvalue (real part):', float(dominant_eigenvalue.real))